# 41 · Transform & semantic — Cube, the semantic layer over the marts

**Cube is the mesh's headless semantic layer: define a metric *once*, query it
*everywhere*.** It sits one layer above the dbt marts — where notebook `20` queried the
marts as raw Iceberg tables through Trino, this notebook queries *governed metrics*
defined over those same marts. You don't write `avg(danceability)` in every dashboard,
app and agent and hope they all wrote it the same way; you define `avg_danceability` once,
in one model, and every consumer asks for that one definition through a single API.

That is the whole thesis of a semantic layer:

> **Define the metric once. Serve it over SQL, REST and GraphQL. Every consumer gets the
> same governed number — no drift, no re-implementation.**

Cube exposes three protocols over the marts (SQL on `:15432`, REST + GraphQL on `:4000`);
this notebook uses the **SQL API**, which speaks the **Postgres wire protocol**, so any
Postgres client reaches it. Under the hood Cube compiles each query to Trino SQL against
`iceberg.dbt.mart_*` (through the `trino-noauth` proxy) — but the consumer never sees
Trino, only the governed cubes.

### The one rule: measures go through `MEASURE()`

A cube has two kinds of column, and they are queried differently:

- **Dimensions** — the group-by columns (`track_genre`, `country`, `year`). Select them
  **normally**, exactly like any SQL column.
- **Measures** — the pre-defined aggregations (`avg_danceability`, `total_plays_sum`).
  These **must be wrapped in `MEASURE(...)`**. A bare `SELECT avg_danceability` is
  *rejected* — the column "could not be resolved from available columns".

That rule is not a quirk to work around; it *is* the semantic layer doing its job. You
cannot accidentally compute a metric a different way than everyone else, because you never
write the aggregation — you name the governed one and Cube supplies the definition. Section
5 demonstrates both halves of the rule against the live API.

### Two ways to reach governed metrics: Cube vs MetricFlow (notebook `40`)

This wave ships **two** semantic options, and they are complementary:

- **MetricFlow** (notebook `40`) — metrics declared *inside the dbt project*, queried with
  `mf query`. dbt-native, CLI-first, scoped here to the time-shaped health marts.
- **Cube** (this notebook) — a *serving* semantic **API**: the same class of governed
  metric, but reachable over SQL / REST / GraphQL by BI tools, apps, agents and notebooks
  alike. The closing section lays out when to reach for which.

> **Read-only, throughout.** Every statement here is a `SELECT` against the SQL API. Cube
> is a serving layer over the marts — there is nothing to write, and no cleanup section
> (as in notebooks `20` / `22`, which also only read).

## Setup

The Cube SQL API is the Postgres wire protocol, so **any** Postgres client reaches it — we
use `psycopg`, which is not in the singleuser base image (it ships `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`). `polars` — used to render result frames, exactly as in
notebooks `20` / `22` — already ships in the image.

In [1]:
%pip install -q "psycopg[binary]"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, the same pattern the query notebooks use. The committed
defaults are the **in-cluster** service DNS (`cube.data-mesh.svc.cluster.local:15432`); a
validation run overrides `CUBE_SQL_HOST` / `CUBE_SQL_PORT` via env **without editing the
notebook**. Cube lives in the istio-injected `data-mesh` namespace, so in-cluster clients
reach it over the mesh.

The SQL API database is always `cube`, the user is `cube`, and the password comes **only**
from `$CUBE_SQL_PASSWORD` — never committed, never printed. We set `autocommit` so that a
deliberately-rejected statement (section 5, and the measure-probe in section 4) leaves the
connection immediately usable rather than in an aborted-transaction state.

`q(sql)` is our tiny helper: run a statement and hand the rows back as a **polars**
DataFrame for display (mirroring notebooks `20` / `22`). We prove the connection by
**listing the cubes**, not by echoing the endpoint or the password.

In [2]:
import os
import psycopg
import polars as pl

conn = psycopg.connect(
    host=os.environ.get("CUBE_SQL_HOST", "cube.data-mesh.svc.cluster.local"),
    port=int(os.environ.get("CUBE_SQL_PORT", "15432")),
    user=os.environ.get("CUBE_SQL_USER", "cube"),
    dbname="cube",
    password=os.environ["CUBE_SQL_PASSWORD"],   # from env only — never committed
    autocommit=True,   # each statement stands alone; a rejected one doesn't poison the next
)

def q(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pl.DataFrame(rows, schema=cols, orient="row")

# prove the connection by what Cube serves — the governed cubes — not by the address
cubes = q("SELECT table_name FROM information_schema.tables "
          "WHERE table_schema = 'public' ORDER BY table_name")
print(f"connected to the Cube SQL API — {cubes.height} cubes served over the marts")

connected to the Cube SQL API — 7 cubes served over the marts

## Discover — the cubes, and each cube's dimensions vs measures

Cube is self-describing over the SQL API's `information_schema`. `information_schema.tables`
lists the **cubes** (each maps to one dbt mart); `information_schema.columns` lists a cube's
**columns**. This is how you explore a semantic model before writing a metric query — you
browse the governed vocabulary, not the underlying tables.

In [3]:
cubes

table_name
str
"""artist_popularity"""
"""country_health"""
"""fma_genre_tree"""
"""genre_audio_profile"""
"""personality_by_country"""
"""spotify_audio"""
"""state_health_trends"""


Seven cubes, one per dbt mart. Now the more interesting half: **within a cube, which
columns are dimensions and which are measures?** This matters because it decides how you
query each one — bare, or wrapped in `MEASURE()`.

`information_schema.columns` alone **can't tell you**: it reports each column's SQL
`data_type`, but a dimension can perfectly well be numeric (`country_health.year` is
`numeric`, yet it's a group-by dimension, not a metric). Type is not role.

The **governed** signal is the `MEASURE()` contract itself: a column **is a measure exactly
when Cube accepts it inside `MEASURE(...)`** — and rejects a dimension there ("Dimension
'year' was used with the aggregate function 'MEASURE()'"). So we classify each column by
probing it once, catching the rejection so no error output is left. (We drop Cube's internal
`__user` / `__cubeJoinField` columns, which aren't part of the model.)

In [4]:
CUBE = "spotify_audio"

cols = q(f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = '{CUBE}'
    ORDER BY ordinal_position
""").filter(~pl.col("column_name").str.starts_with("__"))

def is_measure(cube, col):
    """A column is a measure iff Cube accepts it inside MEASURE(); a dimension is rejected."""
    try:
        q(f"SELECT MEASURE({col}) FROM {cube} LIMIT 1")
        return True
    except Exception:
        return False   # rejected -> it's a dimension (caught, so no error output)

roles = cols.with_columns(
    role=pl.col("column_name").map_elements(
        lambda c: "measure" if is_measure(CUBE, c) else "dimension",
        return_dtype=pl.Utf8,
    )
)
print(f"{CUBE}: "
      f"{roles.filter(pl.col('role')=='dimension').height} dimensions, "
      f"{roles.filter(pl.col('role')=='measure').height} measures")
roles.select(["column_name", "data_type", "role"])

spotify_audio: 2 dimensions, 5 measures


column_name,data_type,role
str,str,str
"""count""","""bigint""","""measure"""
"""avg_danceability""","""numeric""","""measure"""
"""avg_energy""","""numeric""","""measure"""
"""avg_valence""","""numeric""","""measure"""
"""avg_tempo""","""numeric""","""measure"""
"""track_id""","""text""","""dimension"""
"""track_genre""","""text""","""dimension"""


Read that frame as the cube's governed vocabulary: `track_genre` / `track_id` are
**dimensions** you group by; `count` and the `avg_*` columns are **measures** you must call
through `MEASURE()`. Note `count` is a `bigint` and the averages are `numeric` — different
types, same role — which is exactly why the `MEASURE()` probe, not the data type, is the
honest classifier.

## Query the governed measures

Now the payoff — ask each cube for its metric. The shape is always the same: **dimensions
in the `SELECT` and `GROUP BY` normally, measures wrapped in `MEASURE()`**. Three cubes,
three governed metrics, rendered via polars.

**Most danceable genres** — `spotify_audio.avg_danceability` over the `track_genre`
dimension:

In [5]:
q("""
    SELECT track_genre, MEASURE(avg_danceability) AS avg_danceability
    FROM spotify_audio
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 5
""")

track_genre,avg_danceability
str,f64
"""kids""",0.778808
"""chicago-house""",0.76624
"""latino""",0.755487
"""reggaeton""",0.743284
"""minimal-techno""",0.732045


**Longest-lived countries** — `country_health.avg_life_expectancy`, the same governed
metric a dashboard or an agent would get over REST:

In [6]:
q("""
    SELECT country, MEASURE(avg_life_expectancy) AS avg_life_expectancy
    FROM country_health
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""")

country,avg_life_expectancy
str,f64
"""JPN""",83.205168
"""CHE""",81.975752
"""AUS""",81.741702
"""SGP""",81.686158
"""ISL""",81.520811
"""ITA""",81.497205
"""ESP""",81.35448
"""SWE""",81.210192
"""FRA""",81.034385


**Most-played artists** — `artist_popularity.total_plays_sum`. This is the same
answer notebook `20` computed with a raw `SUM(total_plays)` straight off the Iceberg mart —
but here the aggregation is *defined in the model*, so every consumer that asks for
`total_plays_sum` gets identical numbers without re-writing the SQL:

In [7]:
q("""
    SELECT artist_name, MEASURE(total_plays_sum) AS total_plays
    FROM artist_popularity
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""")

artist_name,total_plays
str,f64
"""the beatles""",2.4535627e7
"""radiohead""",2.2155328e7
"""coldplay""",1.3448263e7
"""pink floyd""",1.2849101e7
"""metallica""",1.2309799e7
"""muse""",1.2265731e7
"""nine inch nails""",1.1087033e7
"""red hot chili peppers""",1.0878873e7
"""linkin park""",1.0229522e7


## The `MEASURE()` contract, demonstrated

The one rule, shown against the live API. First the **wrong** way — a bare aggregate column,
the mistake a SQL user reaching for `avg_danceability` directly would make. Cube **rejects**
it. We catch the error and print its message, so this cell *succeeds* and leaves no error
output — the rejection is the lesson, not a failure:

In [8]:
bad = "SELECT track_genre, avg_danceability FROM spotify_audio GROUP BY 1 LIMIT 5"
try:
    q(bad)
    print("UNEXPECTED: Cube accepted a bare measure column")
except Exception as e:
    print("Cube REJECTED the bare measure column (as designed):\n")
    print("  ", str(e).splitlines()[0])
    print("\nThe fix is not to write the aggregation yourself — it is to ask for the")
    print("governed one by name, through MEASURE().")

Cube REJECTED the bare measure column (as designed):

   Planning Error: Initial planning error: Error during planning: Expression #spotify_audio.avg_danceability could not be resolved from available columns: #spotify_audio.track_genre

The fix is not to write the aggregation yourself — it is to ask for the
governed one by name, through MEASURE().


And the **right** way — the identical intent, but the measure named through
`MEASURE()`. Same query, one wrapper, and Cube supplies the governed definition:

In [9]:
q("""
    SELECT track_genre, MEASURE(avg_danceability) AS avg_danceability
    FROM spotify_audio
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 5
""")

track_genre,avg_danceability
str,f64
"""kids""",0.778808
"""chicago-house""",0.76624
"""latino""",0.755487
"""reggaeton""",0.743284
"""minimal-techno""",0.732045


That contrast *is* the semantic layer. The bare form can't compile because there is no
raw `avg_danceability` column to select — there is only a *measure definition*, reachable
solely through `MEASURE()`. You are structurally prevented from computing the metric a
different way than the rest of the platform. Consistency isn't a convention you have to
remember; it's enforced by the API.

## When to reach for Cube

**Reach for Cube when a *non-BI* consumer needs a governed metric, or when one metric
definition must be identical across many tools:**
- **An app, agent or LLM needs a consistent number over an API** — this is Cube's
  distinctive job in the stack. A dashboard, a React app and an agent (Hermes) all ask for
  `country_health.avg_life_expectancy` over SQL / REST / GraphQL and get the *same* governed
  value. Nothing else in the mesh serves metrics to non-BI consumers this way.
- **One definition, many protocols, no drift** — define `avg_danceability` once; SQL
  clients, GraphQL queries and REST calls all compile to the identical Trino aggregation.
  BI tools can consume Cube's SQL API (`:15432`) instead of raw Trino to inherit the same
  governed metrics (Superset does exactly this in the lab).

**Reach for MetricFlow (notebook `40`) instead when the metric lives in the dbt project and
you work from the CLI** — `mf query` against metrics declared in `semantic_models.yml`,
dbt-native and version-controlled alongside the models. Cube and MetricFlow are the two
halves of this semantic wave: MetricFlow defines metrics *in dbt*; Cube *serves* governed
metrics over an API to every kind of consumer.

**Reach for raw Trino (notebook `20`) when you want the mart tables themselves, ad hoc** —
arbitrary SQL, cross-catalog joins, columns the semantic model doesn't expose. Trino gives
you the ungoverned `iceberg.dbt.mart_*` tables and full SQL freedom; Cube gives you the
governed *metrics* over those same marts, and only those.

| you want to… | use |
|--------------|-----|
| a governed metric served to an app / agent / LLM / BI tool over an API | **Cube** (this notebook) |
| metrics defined in the dbt project, queried from the CLI | **MetricFlow** (notebook `40`) |
| arbitrary SQL / joins over the raw mart tables | **Trino** (notebook `20`) |

Cube is the mesh's answer to "everyone must agree on what this number *means*" — one
definition, compiled to Trino, served over three protocols, and enforced by the one rule
that a measure is only ever reachable through `MEASURE()`.